# 04 RAG Demo

Черновик для сборки полного пайплайна: documents -> chunks -> retrieval -> prompt -> answer/evaluation.

In [1]:
import numpy as np


def normalize_rows(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1
    return matrix / norms


def cosine_scores(query_vector: np.ndarray, doc_vectors: np.ndarray) -> np.ndarray:
    query_norm = np.linalg.norm(query_vector)
    if query_norm == 0:
        return np.zeros(doc_vectors.shape[0])
    normalized_query = query_vector / query_norm
    normalized_docs = normalize_rows(doc_vectors)
    with np.errstate(divide='ignore', over='ignore', invalid='ignore'):
        return normalized_docs @ normalized_query


def top_k(scores: np.ndarray, k: int) -> list[int]:
    sorted_ind = np.argsort(scores)[::-1]
    return sorted_ind[:k].tolist()


doc_vectors = np.array([
    [0.9, 0.1, 0.0],  # doc0: машинное обучение
    [0.8, 0.2, 0.0],  # doc1: тоже машинное обучение
    [0.1, 0.9, 0.0],  # doc2: информационная безопасность
    [0.0, 0.8, 0.1],  # doc3: тоже ИБ
    [0.0, 0.1, 0.9],  # doc4: животные
], dtype=float)

query_vector = np.array([1.0, 0.0, 0.0])
scores = cosine_scores(query_vector, doc_vectors)
print(scores)
print(top_k(scores, 3))
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
documents = [
    # 0
    "Градиентный спуск используется для оптимизации параметров модели машинного обучения",

    # 1
    "Метод оптимизации помогает найти минимум функции потерь и улучшить качество модели",

    # 2
    "Нейронная сеть обучается на данных и постепенно изменяет веса между слоями",

    # 3
    "Обучение модели включает подбор параметров на основе ошибки на обучающей выборке",

    # 4
    "Линейная регрессия используется для предсказания числовых значений",

    # 5
    "Классификация помогает отнести объект к одному из заранее известных классов",

    # 6
    "TF-IDF используется для поиска документов по важным словам в коллекции текстов",

    # 7
    "Inverted index хранит список документов, в которых встречается каждое слово",

    # 8
    "Retriever ищет релевантные фрагменты документов по запросу пользователя",

    # 9
    "Vector search ищет похожие по смыслу фрагменты текста с помощью embeddings",

    # 10
    "Embedding превращает текст в числовой вектор для сравнения смысловой близости",

    # 11
    "Cosine similarity показывает насколько направления двух векторов похожи друг на друга",

    # 12
    "RAG использует поиск по документам чтобы передать языковой модели релевантный контекст",

    # 13
    "Если retriever нашел плохой контекст языковая модель может дать неправильный ответ",

    # 14
    "Chunk это небольшой фрагмент документа который передается в модель как часть контекста",

    # 15
    "Keyword search ищет точные совпадения слов в документах",

    # 16
    "Semantic search помогает находить документы даже если запрос и текст написаны разными словами",

    # 17
    "Сотрудник может восстановить доступ к учетной записи через заявку в Service Desk",

    # 18
    "Для разблокировки пользователя необходимо подтвердить личность и сбросить пароль",

    # 19
    "Пароль должен быть достаточно сложным и регулярно обновляться согласно политике безопасности",

    # 20
    "Многофакторная аутентификация снижает риск компрометации учетной записи",

    # 21
    "Система предотвращения утечек данных контролирует передачу конфиденциальной информации",

    # 22
    "DLP решение анализирует содержимое сообщений файлов и сетевого трафика",

    # 23
    "Средства защиты информации должны регулярно обновляться и проверяться администраторами",

    # 24
    "Контроллер домена управляет учетными записями пользователей и политиками безопасности",

    # 25
    "SeDebugPrivilege позволяет процессу отлаживать другие процессы и может быть опасной привилегией",

    # 26
    "Команда secedit позволяет экспортировать параметры локальной политики безопасности Windows",

    # 27
    "TLS используется для защиты соединения между клиентом и сервером",

    # 28
    "Кошка является домашним животным и часто живет рядом с человеком",

    # 29
    "Автомобильный двигатель преобразует энергию топлива в механическое движение"
]
doc_matrix = model.encode(documents)
print(doc_matrix.shape)
query = input("text the query: ")
query_vector = model.encode(query)
scores = cosine_scores(query_vector, doc_matrix)
best = top_k(scores, 3)
for indx in best:
    print(round(scores[indx], 2), documents[indx])


class VectorIndex:
    def __init__(self):
        self.vectors = None
        self.items = []

    def add(self, vectors, items):
        if self.vectors is None:
            self.vectors = vectors
        else:
            self.vectors = np.vstack([self.vectors, vectors])
        self.items.extend(items)

    def search(self, query_vector, k=5, min_score=None):
        scores = cosine_scores(query_vector, self.vectors)
        best = top_k(scores, k)
        results = []
        for i in best:
            if min_score is None or min_score is not None and scores[i] >= min_score:
                results.append((scores[i], self.items[i]))
        return results


index = VectorIndex()
index.add(doc_matrix, documents)
q = model.encode("что такое нейронные сети")
print("without min_score")
for score, item in index.search(q, k=3):
    print(round(score, 2), item)
print("with min_score")
for score, item in index.search(q, k=3, min_score=0.5):
    print(round(score, 2), item)
items = [
    {"text": doc, "source": "ml_notes.txt", "chunk_id": i}
    for i, doc in enumerate(documents)
]

index_meta = VectorIndex()
index_meta.add(doc_matrix, items)

q = model.encode("что такое нейронные сети")
for source, item in index_meta.search(q, k=3):
    print(round(source, 2), "| source:", item["source"], "| chunk_id:", item["chunk_id"])
    print("   ", item["text"])
import re


def tokenize(text: str) -> list[str]:
    text = text.lower()
    text = re.sub(r"[^а-яa-z0-9\- ]", " ", text)
    tokens = text.split()
    return tokens


from collections import Counter


def compute_df(tokenized_documents: list[list[str]]) -> dict[str, int]:
    df = Counter()
    for tokens in tokenized_documents:
        uniq_terms = set(tokens)
        for term in uniq_terms:
            df[term] += 1
    return dict(df)


import math


def compute_idf(df: dict[str, int], total_doc: int) -> dict[str, float]:
    idf = {}
    for term in df:
        idf[term] = math.log(total_doc / df[term])
    return idf


def compute_tfidf(tokens: list[str], idf: dict[str, float]) -> dict[str, float]:
    tf = Counter(tokens)
    tfidf = {}
    for term in tf:
        if term in idf:
            tfidf[term] = tf[term] * idf[term]
    return tfidf


def dot_product(vec1: dict[str, float], vec2: dict[str, float]) -> float:
    score = 0.0
    for term, weight in vec1.items():
        score += weight * vec2.get(term, 0.0)
    return score


def retrieve_tfidf(
        query: str,
        documents: list[str],
        top_k: int = 3,
) -> list[tuple[int, float]]:
    tokenized_documents = [tokenize(doc) for doc in documents]
    df = compute_df(tokenized_documents)
    total_doc = len(documents)
    idf = compute_idf(df, total_doc)
    doc_vectors = [compute_tfidf(tokens, idf) for tokens in tokenized_documents]
    query_tokens = tokenize(query)
    query_vector = compute_tfidf(query_tokens, idf)
    scores = []
    for doc_index, doc_vector in enumerate(doc_vectors):
        score = dot_product(query_vector, doc_vector)
        scores.append((doc_index, score))
        scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]


class Retriever:
    def __init__(self, index, embedder):
        self.index = index
        self.embedder = embedder

    def retrieve(self, query, top_k=5, mode="semantic"):
        if mode == "semantic":
            query_vector = self.embedder.encode(query)
            return self.index.search(query_vector, k=top_k)
        elif mode == "tfidf":
            documents = [it["text"] for it in self.index.items]
            tfidf_results = retrieve_tfidf(query, documents, top_k=top_k)
            return [(score, self.index.items[idx]) for idx, source in tfidf_results]
        else:
            raise ValueError("Invalid mode")


query = "как защитить данные компании от утечки"
retriever = Retriever(index_meta, model)
print("=== SEMANTIC ===")
for score, item in retriever.retrieve(query, top_k=3, mode="semantic"):
    print(round(score, 2), item["text"])

print("=== TF-IDF ===")
for score, item in retriever.retrieve(query, top_k=3, mode="tfidf"):
    print(round(score, 2), item["text"])


def build_prompt(query, results):
    context = ""
    for score, item in results:
        context += f"{item['source']}, {item['chunk_id']}\n{item['text']}\n\n"
    promt = f"""Ты ассистент. Отвечай ТОЛЬКО на основе контекста ниже.
Если в контексте нет ответа, честно скажи: "В документах нет ответа."

Контекст:
{context}
Вопрос: {query}

Ответ:"""
    return promt

[0.99388373 0.9701425  0.11043153 0.         0.        ]
[0, 1, 2]


/Users/anastasia_lu/Documents/Claude/RAG/rag-from-scratch-notes-assistant/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


(30, 384)
0.24 Если retriever нашел плохой контекст языковая модель может дать неправильный ответ
0.22 Chunk это небольшой фрагмент документа который передается в модель как часть контекста
0.19 TF-IDF используется для поиска документов по важным словам в коллекции текстов
without min_score
0.76 Нейронная сеть обучается на данных и постепенно изменяет веса между слоями
0.3 Контроллер домена управляет учетными записями пользователей и политиками безопасности
0.29 Градиентный спуск используется для оптимизации параметров модели машинного обучения
with min_score
0.76 Нейронная сеть обучается на данных и постепенно изменяет веса между слоями
0.76 | source: ml_notes.txt | chunk_id: 2
    Нейронная сеть обучается на данных и постепенно изменяет веса между слоями
0.3 | source: ml_notes.txt | chunk_id: 24
    Контроллер домена управляет учетными записями пользователей и политиками безопасности
0.29 | source: ml_notes.txt | chunk_id: 0
    Градиентный спуск используется для оптимизации параметр

In [8]:
import ollama

class Generator:
    def __init__(self, model = "qwen3:8b"):
        self.model = model

    def answer(self, prompt):
        response = ollama.chat(
            model = self.model,
            messages = [{"role": "user", "content": prompt}]
        )
        return response["message"]["content"]

In [9]:
generator = Generator()

query = "как защитить данные компании от утечки"
results = retriever.retrieve(query, top_k=3, mode="semantic")
prompt = build_prompt(query, results)
print(generator.answer(prompt))

Чтобы защитить данные компании от утечки, необходимо:  
1. Установить и настроить систему предотвращения утечек данных для контроля передачи конфиденциальной информации.  
2. Использовать контроллер домена для управления учетными записями пользователей и политиками безопасности.  
3. Регулярно обновлять и проверять средства защиты информации с помощью администраторов.


In [10]:
query2 = "как приготовить борщ"
results2 = retriever.retrieve(query2, top_k=3)
print(generator.answer(build_prompt(query2, results2)))

В документах нет ответа.


In [11]:
eval_set = [
    # --- простые/смысловые по ML ---
    {"question": "как алгоритм подбирает параметры, двигаясь против наклона функции", "expected_chunk_id": 0},   # градиентный спуск
    {"question": "как уменьшить ошибку модели до минимума",                           "expected_chunk_id": 1},   # метод оптимизации
    {"question": "как искусственная сеть из нейронов учится на примерах",             "expected_chunk_id": 2},   # нейронная сеть
    {"question": "чем предсказывают непрерывное числовое значение",                   "expected_chunk_id": 4},   # линейная регрессия
    {"question": "как отнести объект к одной из категорий",                           "expected_chunk_id": 5},   # классификация

    # --- про устройство RAG/поиска ---
    {"question": "что отвечает за подбор подходящих кусков текста под запрос",        "expected_chunk_id": 8},   # retriever
    {"question": "как текст переводят в набор чисел для сравнения смысла",            "expected_chunk_id": 10},  # embedding
    {"question": "чем измеряют схожесть двух векторов по углу между ними",            "expected_chunk_id": 11},  # cosine similarity
    {"question": "зачем языковой модели передают найденный контекст",                "expected_chunk_id": 12},  # RAG
    {"question": "как называется небольшой кусок документа для подачи в модель",      "expected_chunk_id": 14},  # chunk

    # --- про безопасность (смысловые) ---
    {"question": "куда обратиться, если потерял доступ к рабочему аккаунту",          "expected_chunk_id": 17},  # Service Desk
    {"question": "что снижает риск взлома учётной записи дополнительной проверкой",   "expected_chunk_id": 20},  # MFA
    {"question": "чем компания защищается от утечки конфиденциальных данных",         "expected_chunk_id": 21},  # система предотвращения утечек
    {"question": "как шифруют соединение между клиентом и сервером",                 "expected_chunk_id": 27},  # TLS

    # --- бытовой (проверка на чужой домен) ---
    {"question": "какое животное часто держат дома как питомца",                     "expected_chunk_id": 28},  # кошка
]

In [17]:
def hit_rate_at_k(retriever, eval_set, k, mode="semantic"):
    hits = 0
    for quest in eval_set:
        results = retriever.retrieve(quest["question"], top_k=k, mode=mode)
        found_ids = [item["chunk_id"] for score,item in results]
        if quest["expected_chunk_id"] in found_ids:
            hits += 1
    return hits / len(eval_set)


In [20]:
for k in [1, 3, 5]:
    rate = hit_rate_at_k(retriever, eval_set, k, mode="semantic")
    print(f"hit rate@{k} = {rate}")

hit rate@1 = 0.8
hit rate@3 = 0.9333333333333333
hit rate@5 = 1.0


In [19]:
for k in [1, 3, 5]:
    rate = hit_rate_at_k(retriever, eval_set, k, mode="tfidf")
    print(f"hit rate@{k} = {rate}")

hit rate@1 = 0.6
hit rate@3 = 0.8
hit rate@5 = 0.9333333333333333


In [23]:
def show_misses(retriever, eval_set, k, mode):
    for quest in eval_set:
        results = retriever.retrieve(quest["question"], top_k=k, mode=mode)
        found_ids = [item["chunk_id"] for score,item in results]
        if quest["expected_chunk_id"] not in found_ids:
            print(f"[{mode}] miss: '{quest['question']}'")
            print(f"waited for chunk {quest['expected_chunk_id']}, found {found_ids}")


In [24]:
show_misses(retriever, eval_set, k=1, mode="semantic")
print("---")
show_misses(retriever, eval_set, k=1, mode="tfidf")

[semantic] miss: 'что отвечает за подбор подходящих кусков текста под запрос'
waited for chunk 8, found [16]
[semantic] miss: 'зачем языковой модели передают найденный контекст'
waited for chunk 12, found [13]
[semantic] miss: 'как шифруют соединение между клиентом и сервером'
waited for chunk 27, found [18]
---
[tfidf] miss: 'как алгоритм подбирает параметры, двигаясь против наклона функции'
waited for chunk 0, found [1]
[tfidf] miss: 'как уменьшить ошибку модели до минимума'
waited for chunk 1, found [14]
[tfidf] miss: 'чем предсказывают непрерывное числовое значение'
waited for chunk 4, found [0]
[tfidf] miss: 'что отвечает за подбор подходящих кусков текста под запрос'
waited for chunk 8, found [3]
[tfidf] miss: 'чем компания защищается от утечки конфиденциальных данных'
waited for chunk 21, found [2]
[tfidf] miss: 'какое животное часто держат дома как питомца'
waited for chunk 28, found [14]
